# Rober2vecTM — XLM-RoBERTa (Arabic fine-tuned) + Doc2Vec

**Method:** Concatenates **Doc2Vec** document embeddings (trained on this corpus) with embeddings from **XLM-RoBERTa fine-tuned on Arabic** (`Davlan/xlm-roberta-base-finetuned-arabic`), compresses with an autoencoder, and trains **CombinedTM**.

**Why this method:** This combination tests a different transformer backbone entirely — XLM-RoBERTa is a general multilingual model fine-tuned for Arabic (not Tunisian dialect specifically, and architecturally different from BERT), paired again with corpus-specific Doc2Vec. It's a further comparison point alongside AraBERTopic for how non-TunBERT contextual models perform on this dialect.

**Pipeline:** preprocess → Doc2Vec embeddings (trained on this corpus) → XLM-RoBERTa embeddings (mean-pooled last hidden state) → UMAP-reduce RoBERTa → concatenate with Doc2Vec → autoencoder-compress to 64d → CombinedTM at 7 topics, plus an attempted grid search.

## 0. Setup

In [ ]:
!pip install numpy==1.26.4 gensim==4.3.3 --force-reinstall --upgrade
!pip install transformers umap-learn hdbscan scikit-learn nltk pandas openpyxl contextualized-topic-models tensorflow


## 1. Load the corpus

We start from the raw Tunisian dialect social media corpus. Each row is one post/message; we drop empty rows since they carry no signal for topic modeling.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("../data/TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


## 2. Preprocessing

Tunisian dialect on social media mixes Arabic script, Arabizi (Latin transliteration with digits standing in for Arabic letters, e.g. `3` for `ع`), French loanwords, elongated words for emphasis (`hhhhh`, `mriiiigla`), and noise (URLs, mentions, hashtags, emojis). Standard NLP preprocessing pipelines aren't built for this, so we apply dialect-specific cleaning:

1. Remove custom Tunisian stopwords (functional words with no topical meaning)
2. Normalize character elongation (`mriiiigla` → `mrigla`)
3. Convert Arabizi digits back to their Arabic-letter equivalent (`3` → `a`, `7` → `h`, `5` → `kh`, `9` → `k`, `2` → `a`)
4. Strip URLs, mentions, hashtags, emojis and non-alphanumeric symbols
5. Normalize Arabic letter variants (e.g. `إأآا` → `ا`) so the same word isn't split across multiple spellings
6. Remove stopwords a second time (some appear only after cleaning) and drop any resulting empty lines

In [ ]:
# Read stopwords.txt into a set
with open("../data/tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text):
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    text = remove_numbers(text)

    return text


In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


Check how many documents survived preprocessing:

In [ ]:
print(len(final_texts))


17287


## 3. Topic coherence utilities

We evaluate topic quality with two complementary metrics:
- **C_V coherence** — measures how semantically related the top words of a topic are, based on word co-occurrence in a sliding window (via `gensim`)
- **NPMI** (Normalized Pointwise Mutual Information) — a simpler, more interpretable co-occurrence measure computed directly on document sets, less sensitive to corpus size than raw PMI

Both are computed from `final_texts`, split into tokens.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math

tokenized_docs = [doc.split() for doc in final_texts]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores


## 4. Doc2Vec + XLM-RoBERTa hybrid embedding

Doc2Vec is trained fresh on this corpus (fewer epochs than notebook 5, 50 vs 200, since this was the original setting used for this combination). XLM-RoBERTa is used purely for inference — we mean-pool its last hidden state across tokens to get one vector per document.

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from transformers import AutoTokenizer, AutoModel
import torch
import umap
import numpy as np

processed_texts = final_texts

# 1. Doc2Vec embeddings
tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(processed_texts)]
doc2vec_model = Doc2Vec(vector_size=200, window=10, min_count=5, epochs=50, sample=1e-5, workers=4)
doc2vec_model.build_vocab(tagged_data)
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
doc2vec_embs = np.array([doc2vec_model.dv[str(i)] for i in range(len(tagged_data))])

# 2. XLM-RoBERTa embeddings (fine-tuned on Arabic)
tokenizer = AutoTokenizer.from_pretrained("Davlan/xlm-roberta-base-finetuned-arabic")
model = AutoModel.from_pretrained("Davlan/xlm-roberta-base-finetuned-arabic")

def roberta_embed(texts):
    embs = []
    for t in texts:
        inputs = tokenizer(t, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embs.append(emb)
    return np.array(embs)

roberta_embs = roberta_embed(processed_texts)

# 3. Reduce RoBERTa (768d) to 128d, then combine with Doc2Vec (200d)
umap_model = umap.UMAP(n_components=128, random_state=42)
roberta_reduced = umap_model.fit_transform(roberta_embs)

hybrid_embs = np.concatenate([doc2vec_embs, roberta_reduced], axis=1)


## 5. Compress with an autoencoder

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)


## 6. Build the CTM dataset and train at 7 topics

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM

tp = TopicModelDataPreparation("Davlan/xlm-roberta-base-finetuned-arabic")

training_dataset = tp.fit(
    text_for_contextual=processed_texts,
    text_for_bow=processed_texts,
    custom_embeddings=compressed_embs
)

bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]  # 64

ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=contextual_size,
    n_components=7,
    num_epochs=5,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3
)

ctm.fit(training_dataset)

topics = ctm.get_topic_lists(10)
for i, topic in enumerate(topics):
    print(f"🟢 Topic {i+1}: {', '.join(topic)}")

texts_tokenized = [doc.split() for doc in processed_texts]
dictionary = Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]

topics_list = ctm.get_topic_lists(20)
cv_score = compute_cv_score(topics_list, texts_tokenized, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_list, texts_tokenized)

print("\n📊 Evaluation Metrics:")
print(f"CV (Topic Coherence): {cv_score:.3f}")
print(f"NPMI: {avg_npmi:.3f}")


🟢 Topic 1: ليبيا, العالم, تونس, الدوله, العام, العمل, الشعب, الناس, خدمه, الجمهوريه
🟢 Topic 2: تشوف, الاعتداء, يخلص, يخدمو, شهر, الكلام, القضاء, الامن, المواطنين, بره
🟢 Topic 3: تعرفو, فاهم, قعد, جا, عامله, شهور, الحقايق, مشات, جديده, نجوم
🟢 Topic 4: فسيح, جناته, يسكنه, يجعل, ينعمو, يصبر, اهله, يغفر, فراقه, ورصينه
🟢 Topic 5: ضبط, خالط, جزاهم, معادشي, يعلن, ينهب, اولادهم, لنعسان, سبين
🟢 Topic 6: علي, ناس, المواطن, يخرج, التلقيح, يلزم, انو, التونسي, الشارع, الشيخ
🟢 Topic 7: alah, yarhmou, ili, khali, bled, men, tounes, ضربتو, fama, chab

📊 Evaluation Metrics:
CV (Topic Coherence): 0.542
NPMI: 0.443


## Results

At 7 topics: **CV 0.542, NPMI 0.443**. Topic 7 stands out as the only one mixing Arabizi (Latin-script Tunisian) with Arabic script — likely capturing a cluster of code-switched posts that other topics, built more from Arabic-script vocabulary, don't pick up.

**Note on the grid search:** the original exploration run for the topic-count grid search on this combination was manually interrupted partway through (`KeyboardInterrupt` on the first iteration) and never completed — so unlike the other notebooks in this project, there's no full grid search table here. Re-running the grid search cell below (same pattern as the other notebooks) would fill that gap if you want the full 5–100 topic sweep for this combination.

In [ ]:
# ===============================
# Grid search over topic counts
# ===============================
# Trains a fresh CombinedTM for each candidate number of topics and scores it,
# so we can pick the topic count that yields the most coherent topics.
topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]
results = []

for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

    topics_list = ctm.get_topic_lists(20)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)
